# Deflating the validated text2sql dataset via near-duplicate groups

Deflates `data/text2sql/validated_100_init_sqls.json` (100 validated question/SQL pairs):
near-identical records are grouped and one representative per group is kept.

Two records are near-duplicates iff:

- **same SQL shape** — identical after `sqlglot` masking (literals, columns, tables, aliases,
  aggregates → `AGG`, scalar fns → `FN`, null-guards stripped, repeated select items / `UNION ALL`
  branches collapsed), or
- **paraphrased question** — Qwen3-Embedding-8B cosine distance < 0.1 (≈ bottom 1% of all pairs).

Groups are connected components over those two relations (union-find). Outputs:
`template_report.csv` (per-record shape/group ids) and `deflated_sqls.json`
(one representative per group).


In [1]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import sqlglot
from sqlglot import exp

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "text2sql"

DIALECT = "duckdb"

records = json.loads((DATA_DIR / "validated_117_init_sqls.json").read_text())
df = pd.DataFrame(records)
print(f"{len(df)} records")

117 records


In [2]:
# parse all SQLs once; fail loudly if any does not parse
asts = [sqlglot.parse_one(sql, read=DIALECT) for sql in df["sql"]]

# which physical tables does each query touch?
def physical_tables(ast):
    ctes = {cte.alias for cte in ast.find_all(exp.CTE)}
    return sorted({t.name for t in ast.find_all(exp.Table)} - ctes)

df["tables"] = [physical_tables(a) for a in asts]
Counter(t for ts in df["tables"] for t in ts)

Counter({'outflow': 50, 'wetter': 48, 'swc': 33, 'radiation': 28, 'tsoil': 16})

In [3]:
def canonical(ast):
    return ast.sql(dialect=DIALECT, normalize=True)


def mask_literals(ast):
    def t(node):
        if isinstance(node, exp.Literal):
            return exp.Literal.string("?") if node.is_string else exp.Literal.number("0")
        return node

    return ast.copy().transform(t)


def normalize_aliases(ast):
    """Rename CTE names and table aliases positionally (t1, t2, ...).

    Two passes: collect original names first, then apply — renaming while
    walking would re-map already-renamed aliases.
    """
    ast = ast.copy()
    mapping = {}
    for node in ast.walk():  # CTE names are TableAlias nodes too
        if isinstance(node, exp.TableAlias) and node.name:
            mapping.setdefault(node.name, f"t{len(mapping) + 1}")

    for node in ast.walk():
        if isinstance(node, exp.TableAlias) and node.name in mapping:
            node.set("this", exp.to_identifier(mapping[node.name]))
        elif isinstance(node, exp.Column) and node.table in mapping:
            node.set("table", exp.to_identifier(mapping[node.table]))
        elif isinstance(node, exp.Table) and node.name in mapping:
            node.set("this", exp.to_identifier(mapping[node.name]))
    return ast


def expand_positional_refs(ast):
    """GROUP BY 1 / ORDER BY 2 -> the referenced select expression; ORDER BY alias -> expression.

    Without this, `GROUP BY 1` and `GROUP BY DATE_TRUNC('MONTH', ts)` produce different
    shapes for identical queries.
    """
    ast = ast.copy()
    for select in ast.find_all(exp.Select):
        items = select.expressions
        unalias = lambda e: e.this if isinstance(e, exp.Alias) else e
        aliases = {e.alias: e.this for e in items if isinstance(e, exp.Alias)}

        group = select.args.get("group")
        if group:
            new = []
            for e in group.expressions:
                if isinstance(e, exp.Literal) and not e.is_string:
                    i = int(e.name) - 1
                    if 0 <= i < len(items):
                        e = unalias(items[i]).copy()
                new.append(e)
            group.set("expressions", new)

        order = select.args.get("order")
        if order:
            for o in order.expressions:
                e = o.this
                if isinstance(e, exp.Literal) and not e.is_string:
                    i = int(e.name) - 1
                    if 0 <= i < len(items):
                        o.set("this", unalias(items[i]).copy())
                elif isinstance(e, exp.Column) and not e.table and e.name in aliases:
                    o.set("this", aliases[e.name].copy())
    return ast


def mask_columns(ast):
    ast = ast.copy()
    for node in ast.walk():
        if isinstance(node, exp.Column):
            node.set("this", exp.to_identifier("col"))
        elif isinstance(node, exp.Alias) and isinstance(node.args.get("alias"), exp.Identifier):
            node.set("alias", exp.to_identifier("col"))
    return ast


def mask_tables(ast):
    ast = ast.copy()
    for node in ast.walk():
        if isinstance(node, exp.Table) and isinstance(node.this, exp.Identifier):
            node.set("this", exp.to_identifier("tbl"))
    return ast


# --- shape-level transforms: differences that don't change the query plan family ---


def strip_null_guards(ast):
    """COALESCE(x, <lit>) -> x, NULLIF(x, <lit>) -> x: defensive wrappers, not structure."""
    def t(node):
        if isinstance(node, exp.Coalesce) and all(
            isinstance(e, exp.Literal) for e in node.expressions
        ):
            return node.this
        if isinstance(node, exp.Nullif) and isinstance(node.expression, exp.Literal):
            return node.this
        return node

    return ast.copy().transform(t)


def mask_functions(ast):
    """Aggregate functions -> AGG(col), scalar functions (incl. casts) -> FN(col)."""
    def t(node):
        if isinstance(node, exp.Anonymous) and node.name in ("AGG", "FN"):
            return node
        if isinstance(node, exp.AggFunc):
            return exp.Anonymous(this="AGG", expressions=[exp.column("col")])
        if isinstance(node, exp.Func):
            return exp.Anonymous(this="FN", expressions=[exp.column("col")])
        return node

    return ast.copy().transform(t)


def drop_column_aliases(ast):
    """AGG(col) AS col -> AGG(col): after masking, aliases carry no information, and only the
    first branch of a UNION keeps aliases in canonical SQL — they'd block branch dedup."""
    def t(node):
        if isinstance(node, exp.Alias):
            return node.this
        return node

    return ast.copy().transform(t)


def dedupe_select_items(ast):
    """Collapse select items identical after masking (AVG per roof x5 -> x1)."""
    ast = ast.copy()
    for select in ast.find_all(exp.Select):
        seen, new = set(), []
        for e in select.expressions:
            s = e.sql(dialect=DIALECT)
            if s not in seen:
                seen.add(s)
                new.append(e)
        select.set("expressions", new)
    return ast


def dedupe_setop_branches(ast):
    """Collapse UNION/UNION ALL branches identical after masking (one SELECT per roof type ->
    a single branch, regardless of roof count). Fixpoint loop: branches collapse pairwise."""
    ast = ast.copy()
    while True:
        def t(node):
            if isinstance(node, exp.SetOperation) and node.this.sql(
                dialect=DIALECT
            ) == node.expression.sql(dialect=DIALECT):
                return node.this
            return node

        new = ast.transform(t)
        if new.sql(dialect=DIALECT) == ast.sql(dialect=DIALECT):
            return new
        ast = new


In [4]:
def shape(ast):
    a = expand_positional_refs(normalize_aliases(ast))
    a = strip_null_guards(a)
    a = mask_functions(mask_columns(mask_literals(a)))
    a = drop_column_aliases(a)
    a = dedupe_select_items(a)
    a = dedupe_setop_branches(a)
    return canonical(mask_tables(a))


df["shape"] = [shape(a) for a in asts]

print(f"{df['shape'].nunique()} unique shapes\n")
print(f"example: {df['shape'].iloc[0]}")


99 unique shapes

example: WITH t3 AS (SELECT FN(col), AGG(col) FROM tbl GROUP BY FN(col)), t4 AS (SELECT FN(col), AGG(col) FROM tbl GROUP BY FN(col)) SELECT AGG(col) FROM tbl AS t1 JOIN tbl AS t2 ON t1.col = t2.col WHERE FN(col)


## Near-duplicate groups: exact shape ∪ question paraphrases (Used to actually deflate data)

Exact shape templates miss pairs whose SQL differs slightly but whose **question** is a
wording-level paraphrase. We embed the questions with **Qwen3-Embedding-8B** served on Blablador
(the model doesn't fit the local 6 GB GPU; one batch of API calls, cached to disk) and join
records that are either:

- **same shape** (identical masked SQL), or
- **question cosine distance < 0.1** — the bottom ~1% of all pairs, i.e. near-identical wording.

Why not TF-IDF / embedding hierarchical clustering (the previous approach)? Diagnostics below
show that in this single-domain dataset question distances carry almost **no signal about
structural duplication**: pairs with identical SQL shape have the *same* median question distance
as random pairs. Any linkage criterion therefore chains the whole "roof × weather" topic into
40-member blobs — the old 28-cluster / 43-kept result was an artifact of char-n-gram noise, not
real redundancy.


In [5]:
import os

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

EMB_MODEL = "openai/alias-qwen3-8b-embeddings"  # Qwen3-Embedding-8B on Blablador
EMB_CACHE = DATA_DIR / "question_embeddings.npz"

questions = df["question"].tolist()
Q = None
if EMB_CACHE.exists():
    z = np.load(EMB_CACHE)
    if list(z["questions"]) == questions:
        Q = z["emb"]
if Q is None:
    import litellm

    vecs = []
    for i in range(0, len(questions), 16):
        r = litellm.embedding(
            model=EMB_MODEL,
            input=questions[i : i + 16],
            api_base=os.environ["LLM_API_BASE_BLABLADOR"],
            api_key=os.environ["LLM_API_KEY_BLABLADOR"],
        )
        vecs.extend(d["embedding"] for d in r.data)
    Q = np.array(vecs)
    np.savez_compressed(EMB_CACHE, emb=Q, questions=np.array(questions))
print(f"question embeddings: {Q.shape}")

question embeddings: (117, 4096)


In [6]:
import itertools

from sklearn.metrics.pairwise import cosine_distances

D_q = cosine_distances(Q)
pairs = list(itertools.combinations(range(len(df)), 2))
same_shape = [(i, j) for i, j in pairs if df["shape"][i] == df["shape"][j]]

pct = [0, 5, 25, 50, 75, 100]
dq = lambda ps: np.percentile([D_q[i, j] for i, j in ps], pct).round(3)
diag = pd.DataFrame([dq(pairs), dq(same_shape)], index=["all pairs", "same-shape pairs"],
                    columns=[f"p{p}" for p in pct])
print(f"{len(same_shape)} same-shape pairs out of {len(pairs)}")
diag  # same-shape median ≈ all-pairs median -> question distance can't rank structural dupes

29 same-shape pairs out of 6786


,p0,p5,p25,p50,p75,p100
all pairs,0.035,0.185,0.292,0.370,0.437,0.707
same-shape pairs,0.115,0.125,0.186,0.277,0.375,0.446


In [7]:
Q_THRESHOLD = 0.10  # cosine distance: ≈ bottom 1% of pairs, wording-level paraphrases only

parent = list(range(len(df)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


for i, j in pairs:
    if df["shape"][i] == df["shape"][j] or D_q[i, j] < Q_THRESHOLD:
        parent[find(i)] = find(j)

df["group"] = pd.factorize(np.array([find(i) for i in range(len(df))]))[0]
group_sizes = df["group"].value_counts()
print(f"{group_sizes.size} near-duplicate groups, "
      f"{int(group_sizes[group_sizes > 1].sum())} records in groups>1, "
      f"largest {group_sizes.max()}")
group_sizes.value_counts().sort_index().rename_axis("group size").to_frame("# groups")

75 near-duplicate groups, 58 records in groups>1, largest 10


,# groups
group size,
1,59
2,7
3,2
4,4
5,1
7,1
10,1


In [8]:
# inspect all groups: multi-member ones are the near-duplicates that will be collapsed
for gid, sub in df.groupby("group"):
    if len(sub) > 1:
        reason = "same shape" if sub["shape"].nunique() < len(sub) else "question paraphrase"
        print(f"=== group {gid} ({len(sub)} members, {reason}) ===")
    else:
        print(f"=== group {gid} (singleton) ===")
    for _, row in sub.iterrows():
        print(f"  [{row.name}] {row['question']}")
    print()

=== group 0 (2 members, same shape) ===
  [0] How many days had runoff from the smart irrigated extensive green roof equal to at least 80% of the recorded precipitation?
  [1] How often did the non-irrigated extensive green roof produce less daily outflow than the smart irrigated extensive green roof on days with at least 10 mm precipitation?

=== group 1 (3 members, question paraphrase) ===
  [2] Which roof type had the lowest median daily outflow on days with at least 15 mm precipitation?
  [6] Which roof type most frequently had the highest daily outflow among the monitored lysimeters?
  [91] Which roof type had the highest number of days with daily outflow above 10 liters?

=== group 2 (singleton) ===
  [3] How did the daily runoff-to-precipitation ratio differ between the gravel roof and the wetland green roof on days with at least 20 mm precipitation?

=== group 3 (4 members, question paraphrase) ===
  [4] On days with at least 15 mm precipitation, what percentage of precipitatio

## Outputs

- `template_report.csv` — per-record shape id + group id, for manual review.
- `deflated_sqls.json` — one representative per **near-duplicate group** (the meaningful dedup
  unit: same shape or paraphrased question).


In [9]:
# per-record report
df["shape_id"] = df.groupby("shape", sort=False).ngroup()
df["group_size"] = df["group"].map(group_sizes)

report = df[["question", "sql", "tables", "shape_id", "group", "group_size"]]
report.to_csv(DATA_DIR / "template_report.csv", index_label="idx")

# deflated dataset: first record of each near-duplicate group
deflated_idx = df.drop_duplicates(subset="group").index
deflated = [records[i] for i in deflated_idx]
(DATA_DIR / "deflated_sqls.json").write_text(json.dumps(deflated, indent=2, ensure_ascii=False))
print(f"deflated: {len(deflated)} / {len(df)} records kept")

deflated: 75 / 117 records kept


In [10]:
# sanity: outputs reload cleanly, one representative per group, all questions unique
deflated_reloaded = json.loads((DATA_DIR / "deflated_sqls.json").read_text())
assert len(deflated_reloaded) == df["group"].nunique()

reloaded = pd.read_csv(DATA_DIR / "template_report.csv")
assert len(reloaded) == len(df)

deflated_qs = {r["question"] for r in deflated_reloaded}
assert len(deflated_qs) == len(deflated_reloaded)
assert deflated_qs <= set(df["question"])
print("sanity checks passed")

sanity checks passed
